# SIAO-CNN-ORNN Training on Google Colab

**Nuclear Power Plant Accident Detection**

This notebook runs the full training pipeline on Google Colab with GPU acceleration.

## Pipeline
```
Data → Sliding Windows → CNN (pre-trained, 512-dim) + Statistical (485-dim) + WKS (97-dim) → SIAO → ORNN (224-dim) → Backprop → Softmax
```

## Requirements
- Google Colab with GPU runtime (T4 or better)
- Public repo: `sisoodiya/SIAO-CNN-ORNN`

## 1. Setup Environment

In [ ]:
# Verify GPU is available
!nvidia-smi

In [ ]:
# Clone the repository
!git clone https://github.com/sisoodiya/SIAO-CNN-ORNN.git
%cd SIAO-CNN-ORNN

In [ ]:
# Install dependencies (Colab already has torch, numpy, sklearn, matplotlib)
!pip install -q imbalanced-learn optuna rich tqdm seaborn scipy

In [ ]:
import torch
import numpy as np
import os

# Verify setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# Verify data exists
data_dir = 'data/Operation_csv_data/'
classes = os.listdir(data_dir)
print(f"\nData directory: {data_dir}")
print(f"Classes found: {len(classes)}")
for c in sorted(classes):
    n_files = len([f for f in os.listdir(os.path.join(data_dir, c)) if f.endswith('.csv')])
    print(f"  {c}: {n_files} samples")

## 2. Load & Explore Data

In [ ]:
from src.siao_cnn_ornn.data.nppad_loader import NPPADDataPipeline, CLASS_NAMES

pipeline = NPPADDataPipeline(
    data_dir=data_dir,
    max_timesteps=100
)

X_raw, y_raw = pipeline.run(use_cache=True)

print(f"Dataset shape: {X_raw.shape}")
print(f"Samples: {X_raw.shape[0]}")
print(f"Timesteps: {X_raw.shape[1]}")
print(f"Features: {X_raw.shape[2]}")
print(f"Classes: {len(np.unique(y_raw))}")

In [ ]:
import matplotlib.pyplot as plt

unique, counts = np.unique(y_raw, return_counts=True)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar([CLASS_NAMES.get(u, str(u)) for u in unique], counts, color='steelblue')
ax.set_xlabel('Accident Type')
ax.set_ylabel('Samples')
ax.set_title('NPPAD Class Distribution')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f"\nMin samples: {counts.min()} | Max: {counts.max()} | Imbalance ratio: {counts.max()/counts.min():.1f}:1")

## 3. Train Model (Full Pipeline)

In [ ]:
from train_pipeline import run_complete_pipeline

# Optuna-optimized hyperparameters + GPU acceleration
# Model sized for 1,237 samples (bigger = overfitting + slow SIAO search)
# Speed comes from AMP, batch SIAO eval, large batch size — not bigger model
config = {
    'data_dir': data_dir,
    'window_size': 100,
    'stride': 25,
    'cnn_embedding_dim': 512,      # Optuna-tuned
    'wks_dim': 97,
    'rnn_hidden_size': 224,        # Optuna-tuned (bigger = overfits 1237 samples)
    'rnn_num_layers': 2,           # Optuna-tuned
    'num_classes': None,           # auto-detected
    'n_folds': 10,
    'test_size': 0.2,
    'wks_pop_size': 15,            # Optuna-tuned
    'wks_max_iter': 30,            # Optuna-tuned
    'siao_pop_size': 30,           # balanced: enough search agents
    'siao_max_iter': 100,          # balanced: enough iterations
    'bp_epochs': 300,              # longer training (early stopping guards)
    'bp_lr': 0.00157,              # Optuna-tuned
    'fc_dropout': 0.164,           # Optuna-tuned
    'weight_decay': 1.97e-05,      # Optuna-tuned
    'batch_size': 512,             # large batch = better GPU utilization
    'save_dir': 'results/'
}

print("Training Configuration:")
for k, v in config.items():
    print(f"  {k}: {v}")

In [ ]:
results = run_complete_pipeline(**config)

print(f"\n{'='*60}")
print(f"Average Accuracy: {results['avg_accuracy']*100:.2f}%")
print(f"Std: {results['std_accuracy']*100:.2f}%")
print(f"Fold scores: {[f'{x*100:.2f}%' for x in results['fold_accuracies']]}")
print(f"{'='*60}")

## 4. Results

In [ ]:
import glob
from PIL import Image

# Display training curves and confusion matrices
plot_files = sorted(glob.glob('results/plots/*.png'))

if plot_files:
    for pf in plot_files:
        img = Image.open(pf)
        fig, ax = plt.subplots(figsize=(10, 8))
        ax.imshow(img)
        ax.set_title(os.path.basename(pf))
        ax.axis('off')
        plt.tight_layout()
        plt.show()
else:
    print("No plots generated yet.")

In [ ]:
# Summary
print(f"\nFold Accuracies:")
for i, acc in enumerate(results['fold_accuracies']):
    bar = '#' * int(acc * 50)
    print(f"  Fold {i+1:2d}: {acc*100:6.2f}% |{bar}")
print(f"\n  Mean:    {results['avg_accuracy']*100:6.2f}% +/- {results['std_accuracy']*100:.2f}%")

## 5. Download Results

In [ ]:
# Zip results for download
!zip -r results.zip results/

from google.colab import files
files.download('results.zip')